# Hansen Ch.11 Multivariate Regression — 习题解答

**Chapter 11 Multivariate Regression**

完整证明与**面向初学者的详细注释**见同目录 `Hansen_Ch11_Exercises_Solutions.md`（强烈建议先读 §0、§1）。

本 notebook 用模拟核对 **Exercise 11.15**：共同 $X$ 下 $(\hat\beta_1,\hat\beta_2)$ 的联合协方差。

> **写给只学过李子奈/陈强的同学：** 本章 = 把 $m$ 个方程**堆叠**成一个大系统，于是第 4 章（GLS）+ 第 7 章（渐近正态、夹心方差）整体照搬，只是矩阵变大、多了 Kronecker 结构。核心新东西是 **SUR（似无关回归）= 系统的 GLS**：
> - 回归元不同 + 跨方程误差相关时，SUR **比逐方程 OLS 更有效**（MC 验证：方差可减半）。
> - **SUR = OLS** 当且仅当：所有方程回归元相同，**或**跨方程误差不相关（$\Sigma$ 对角）。
> - 跨方程函数（如 $\beta_1-\beta_2$）的推断必须用**联合**方差，**不能漏掉跨方程协方差项**。


In [ ]:

import numpy as np

rng = np.random.default_rng(0)
n, k = 5000, 2
# X ~ N(0,I), e ~ bivariate with correlation
X = rng.normal(size=(n, k))
X = np.column_stack([X, np.ones(n)])  # add intercept? use k=2 without intercept for simplicity
X = rng.normal(size=(n, 2))
beta1 = np.array([1.0, -0.5])
beta2 = np.array([0.2, 0.8])
# correlated errors
z = rng.normal(size=(n, 2))
# cov [[1, 0.6],[0.6, 1]]
L = np.array([[1.0, 0.0], [0.6, np.sqrt(1-0.36)]])
e = z @ L.T
Y1 = X @ beta1 + e[:, 0]
Y2 = X @ beta2 + e[:, 1]

Q = X.T @ X / n
b1 = np.linalg.solve(X.T @ X, X.T @ Y1)
b2 = np.linalg.solve(X.T @ X, X.T @ Y2)

# theoretical avar blocks (homosk): Sigma_jl * Qinv / n
Sig = np.cov(e.T, bias=True)
Qinv = np.linalg.inv(Q)
# Monte Carlo many reps for sampling var of sqrt(n)(b-beta)
B = 400
rng2 = np.random.default_rng(1)
diffs = []
for _ in range(B):
    X = rng2.normal(size=(n, 2))
    z = rng2.normal(size=(n, 2))
    e = z @ L.T
    Y1 = X @ beta1 + e[:, 0]
    Y2 = X @ beta2 + e[:, 1]
    b1 = np.linalg.solve(X.T @ X, X.T @ Y1)
    b2 = np.linalg.solve(X.T @ X, X.T @ Y2)
    diffs.append(np.concatenate([np.sqrt(n)*(b1-beta1), np.sqrt(n)*(b2-beta2)]))
S = np.cov(np.array(diffs).T)
# theory: kron(Sigma, Q^{-1}) but Q random; use E[Q]=I
V_theory = np.kron(np.array([[1, 0.6], [0.6, 1]]), np.eye(2))
print("MC cov of sqrt(n)(beta_hat - beta) [approx]:\n", np.round(S, 3))
print("Theory Sigma ⊗ I_2:\n", V_theory)
print("Frobenius relative error:", np.linalg.norm(S - V_theory) / np.linalg.norm(V_theory))


## 理论题索引

| 题号 | 内容 |
|:----:|------|
| 11.1–11.6 | $\Omega,Q,V_\beta$ 简化与 Thm 11.1 |
| 11.7–11.8 | Delta 法、协方差一致 |
| 11.9–11.13 | SUR/GLS 与相对有效性 |
| 11.14 | 生成回归元 $\hat\pi$ 两步估计 |
| 11.15 | 两方程联合推断（本 notebook 模拟） |


## 理论结论的蒙特卡洛验证（无需外部数据）

以下单元格核对 ch11 的 SUR 头条结论：(A) 回归元不同 + 跨方程误差相关时 SUR 比 OLS 更有效；(B) 共同回归元时 SUR = OLS。可独立运行。

In [ ]:
import numpy as np
rng = np.random.default_rng(11)

n, reps = 4000, 3000
rho = 0.7
L = np.array([[1.0, 0.0], [rho, np.sqrt(1 - rho**2)]])          # 误差协方差 [[1,ρ],[ρ,1]] 的 Cholesky
b1, b2 = 1.0, 0.5

# (A) 回归元不同 + 误差相关: SUR > OLS（借用跨方程信息）
ols1, sur1 = [], []
for r in range(reps):
    X1 = rng.standard_normal(n)
    X2 = rng.standard_normal(n) + 0.3 * X1                      # 不同回归元(但相关)
    z = rng.standard_normal((n, 2)); e = z @ L.T
    Y1 = b1 * X1 + e[:, 0]; Y2 = b2 * X2 + e[:, 1]
    ols1.append(np.sum(X1*Y1) / np.sum(X1**2))
    Sinv = np.linalg.inv(np.cov(e.T))                           # 已知 Σ 的 GLS = SUR
    A = np.array([[np.sum(X1**2)*Sinv[0,0], np.sum(X1*X2)*Sinv[0,1]],
                  [np.sum(X1*X2)*Sinv[1,0], np.sum(X2**2)*Sinv[1,1]]])
    rhs = np.array([Sinv[0,0]*np.sum(X1*Y1) + Sinv[0,1]*np.sum(X1*Y2),
                    Sinv[1,0]*np.sum(X2*Y1) + Sinv[1,1]*np.sum(X2*Y2)])
    sur1.append(np.linalg.solve(A, rhs)[0])
print(f"[11.12] 回归元不同+误差相关: var(OLS)={np.var(ols1):.5f} > var(SUR)={np.var(sur1):.5f}  (SUR 更有效)")

# (B) 共同回归元: SUR = OLS（逐样本数值相同）
ols1c, sur1c = [], []
for r in range(reps):
    X = rng.standard_normal(n)
    z = rng.standard_normal((n, 2)); e = z @ L.T
    Y1 = b1 * X + e[:, 0]; Y2 = b2 * X + e[:, 1]
    ols1c.append(np.sum(X*Y1) / np.sum(X**2))
    Sinv = np.linalg.inv(np.cov(e.T))
    A = np.array([[np.sum(X**2)*Sinv[0,0], np.sum(X**2)*Sinv[0,1]],
                  [np.sum(X**2)*Sinv[1,0], np.sum(X**2)*Sinv[1,1]]])
    rhs = np.array([Sinv[0,0]*np.sum(X*Y1) + Sinv[0,1]*np.sum(X*Y2),
                    Sinv[1,0]*np.sum(X*Y1) + Sinv[1,1]*np.sum(X*Y2)])
    sur1c.append(np.linalg.solve(A, rhs)[0])
print(f"[11.5/11.6] 共同回归元: var(OLS)={np.var(ols1c):.6f} = var(SUR)={np.var(sur1c):.6f}, "
      f"逐样本|OLS-SUR|均值={np.mean(np.abs(np.array(ols1c) - np.array(sur1c))):.1e}  (SUR = OLS)")